<div dir="rtl">

# نوت‌بوک ۴: ممیزی متن

**فاز ۱، تسک ۷**

متن نقد **وجه اصلی** پروژه است: جنبه‌ها (باتری، دوام، برد کنترل و ...) از آن استخراج می‌شوند و گزارش نهایی از همین جنبه‌ها ساخته می‌شود. سند برای ممیزی متن چهار چیز می‌خواهد:

۱. توزیع طول نقدها
۲. فراوانی نقدهای خیلی کوتاه و خیلی بلند
۳. زبان یا خط غیرمعمول
۴. رابطه‌ی اولیه‌ی متن با امتیاز

**قبل از هر شمارشی**، نشانه‌گذاری فنی (تگ HTML، کد ویدیو) که در نوت‌بوک ۲ پیدا کردیم پاک می‌شود، چون جزو حرف مشتری نیست. تابع پاک‌سازی در `src/text_utils.py` است تا فاز ۲ هم دقیقاً از همان استفاده کند. **فایل داده تغییر نمی‌کند؛** پاک‌سازی فقط در حافظه انجام می‌شود.

</div>

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from text_utils import clean_text, script_of, word_count

cfg = yaml.safe_load(open(ROOT / "configs" / "config.yaml", encoding="utf-8"))
PROC = ROOT / cfg["data"]["processed_dir"]
FIG = ROOT / "figures"
SEED = cfg["seed"]

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

BAR = "#3B6EA8"
NEG, POS = "#B5523B", "#3B6EA8"   # 1-2 stars vs 4-5 stars
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e6e6", "grid.linewidth": 0.6,
    "axes.axisbelow": True, "font.size": 9,
})

reviews = pd.read_parquet(PROC / "reviews_rc.parquet",
                          columns=["rating", "title", "text", "parent_asin", "user_id", "timestamp"])
# exact duplicates are removed for the audit (decision from notebook 2)
reviews = reviews.drop_duplicates().reset_index(drop=True)

reviews["clean"] = reviews["text"].map(clean_text)
reviews["words"] = reviews["clean"].map(word_count)
reviews["chars"] = reviews["clean"].str.len()
reviews["sentiment_group"] = pd.cut(reviews["rating"], [0, 2, 3, 5],
                                    labels=["negative (1-2)", "neutral (3)", "positive (4-5)"])
print(f"reviews after removing exact duplicates: {len(reviews):,}")

<div dir="rtl">

## ۱. توزیع طول نقدها

طول را با **تعداد کلمه** می‌سنجیم، نه تعداد حرف؛ چون «کلمه» به واحد معنا نزدیک‌تر است. نمودار تا ۲۰۰ کلمه را نشان می‌دهد و نقدهای بلندتر در آخرین ستون جمع شده‌اند.

</div>

In [ ]:
w = reviews["words"]
print(w.describe(percentiles=[.1, .25, .5, .75, .9, .99]).round(1).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
cap = 200   # reviews longer than this are grouped into the last bar
ax.hist(w.clip(upper=cap), bins=np.arange(0, cap + 6, 5), color=BAR)
ax.axvline(w.median(), color="#333", lw=1, ls="--")
ax.text(w.median() + 3, ax.get_ylim()[1] * 0.9, f"median = {w.median():.0f} words", fontsize=8.5)
ax.text(cap - 2, ax.get_ylim()[1] * 0.25, f"{(w > cap).mean():.1%}\nlonger\nthan {cap}", ha="right", fontsize=8, color="#555")
ax.set_xlabel(f"words per review (bins of 5; last bar = more than {cap})")
ax.set_ylabel("reviews")
ax.set_title("Review length", loc="left")
plt.tight_layout()
plt.savefig(FIG / "fig05_review_length.png", dpi=200, bbox_inches="tight")
plt.show()

<div dir="rtl">

## ۲. نقدهای خیلی کوتاه و خیلی بلند

برای سؤال ما، نقدی به درد می‌خورد که **دست‌کم یک جنبه** را نام ببرد. «Great» هیچ جنبه‌ای ندارد؛ «Battery dies after 5 minutes» دارد. پس مرزها را این‌طور تعریف می‌کنیم:

</div>

In [ ]:
bands = pd.cut(w, [-1, 0, 3, 9, 50, 200, 500, np.inf],
               labels=["0 (empty)", "1-3", "4-9", "10-50", "51-200", "201-500", "500+"])
band_table = bands.value_counts().sort_index().to_frame("reviews")
band_table["share_%"] = (100 * band_table["reviews"] / len(reviews)).round(2)
band_table

In [ ]:
rng = np.random.default_rng(SEED)
for label in ["1-3", "4-9", "500+"]:
    sample = reviews.loc[bands == label, "clean"]
    picks = sample.sample(min(6, len(sample)), random_state=SEED)
    print(f"--- {label} words ---")
    for t in picks:
        print("  ", t[:150] + ("..." if len(t) > 150 else ""))

<div dir="rtl">

## ۳. زبان و خط

دو بررسی:

- **خط:** حروف هر نقد لاتین است، یا سیریلیک، عربی، چینی-ژاپنی-کره‌ای؟
- **زبان در خط لاتین:** اسپانیایی و فرانسوی هم لاتین‌اند. یک روش ساده: در یک نقد انگلیسی معمولی، بخش زیادی از کلمه‌ها «کلمات پرتکرار» انگلیسی‌اند (the، and، it، is، ...). اگر نقدی دست‌کم ۸ کلمه داشته باشد ولی کمتر از ۱۰٪ کلماتش از این فهرست باشد، احتمالاً انگلیسی نیست.

این روش تقریبی است و فقط برای **تخمین اندازه‌ی مسئله** به کار می‌رود.

</div>

In [ ]:
reviews["script"] = reviews["clean"].map(script_of)
print(reviews["script"].value_counts().to_string())

In [ ]:
stop = set(ENGLISH_STOP_WORDS)

def stopword_share(text):
    words = [t.lower() for t in text.split()]
    return np.mean([t.strip(".,!?;:()\"'") in stop for t in words]) if words else np.nan

latin_long = reviews[(reviews["script"] == "latin") & (reviews["words"] >= 8)]
share = latin_long["clean"].map(stopword_share)
likely_non_english = latin_long[share < 0.10]
print(f"Latin-script reviews with >= 8 words: {len(latin_long):,}")
print(f"likely not English (< 10% English stopwords): {len(likely_non_english):,} "
      f"({100 * len(likely_non_english) / len(reviews):.2f}% of all reviews)")
print()
for t in likely_non_english["clean"].sample(min(8, len(likely_non_english)), random_state=SEED):
    print("  ", t[:140])

<div dir="rtl">

## ۴. رابطه‌ی متن با امتیاز

### ۴-۱. آیا مشتری‌های ناراضی طولانی‌تر می‌نویسند؟

</div>

In [ ]:
by_rating = reviews.groupby("rating")["words"].agg(
    reviews="size", median_words="median", mean_words="mean",
    share_short=lambda s: (s <= 3).mean() * 100).round(1)
by_rating

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4))
data = [reviews.loc[reviews["rating"] == r, "words"].clip(lower=1) for r in range(1, 6)]
ax.boxplot(data, tick_labels=[str(r) for r in range(1, 6)], showfliers=False,
           patch_artist=True, widths=0.5,
           boxprops=dict(facecolor="#dbe5f1", edgecolor=BAR),
           medianprops=dict(color=BAR, lw=2),
           whiskerprops=dict(color="#888"), capprops=dict(color="#888"))
ax.set_yscale("log")
ax.set_xlabel("stars")
ax.set_ylabel("words per review (log scale)")
ax.set_title("5-star reviews are the shortest", loc="left")
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.savefig(FIG / "fig06_length_by_rating.png", dpi=200, bbox_inches="tight")
plt.show()

<div dir="rtl">

### ۴-۲. چه کلمه‌هایی نقدهای منفی را از مثبت جدا می‌کنند؟

برای هر کلمه حساب می‌کنیم در چند درصد نقدهای منفی (۱ و ۲ ستاره) و چند درصد نقدهای مثبت (۴ و ۵ ستاره) آمده است. **نسبت این دو** (به‌شکل لگاریتمی، `log2`) نشان می‌دهد کلمه چقدر مشخصه‌ی یک گروه است:

- عدد **مثبت**: کلمه در نقدهای منفی بیشتر است. مثلاً `+2` یعنی ۴ برابر بیشتر.
- عدد **منفی**: کلمه در نقدهای مثبت بیشتر است.

کلمات بسیار پرتکرار (the، and) و کلماتی که در کمتر از ۲۰۰ نقد آمده‌اند کنار گذاشته می‌شوند. این یک **نگاه اکتشافی** است، نه روش نهایی استخراج جنبه (آن کار فاز ۲ است).

</div>

In [ ]:
polar = reviews[reviews["rating"] != 3]
is_neg = (polar["rating"] <= 2).to_numpy()

vec = CountVectorizer(stop_words="english", min_df=200, binary=True,
                      token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b", lowercase=True)
X = vec.fit_transform(polar["clean"])
vocab = np.array(vec.get_feature_names_out())

neg_rate = (np.asarray(X[is_neg].sum(axis=0)).ravel() + 1) / (is_neg.sum() + 2)
pos_rate = (np.asarray(X[~is_neg].sum(axis=0)).ravel() + 1) / ((~is_neg).sum() + 2)
words = pd.DataFrame({
    "word": vocab,
    "in_negative_%": (100 * neg_rate).round(2),
    "in_positive_%": (100 * pos_rate).round(2),
    "log2_ratio": np.log2(neg_rate / pos_rate).round(2),
}).set_index("word")
print(f"vocabulary size: {len(vocab):,} words")

In [ ]:
top_neg = words.nlargest(25, "log2_ratio")
top_pos = words.nsmallest(25, "log2_ratio")
display(top_neg)
display(top_pos)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 6))
for ax, df, color, title in [
    (axes[0], top_neg.iloc[::-1], NEG, "More common in 1-2 star reviews"),
    (axes[1], top_pos.iloc[::-1], POS, "More common in 4-5 star reviews"),
]:
    ax.barh(df.index, df["log2_ratio"].abs(), color=color, height=0.65)
    ax.set_title(title, loc="left")
    ax.set_xlabel("|log2 ratio|  (1 = twice as common, 2 = 4x)")
    ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.savefig(FIG / "fig07_distinctive_words.png", dpi=200, bbox_inches="tight")
plt.show()

<div dir="rtl">

### ۴-۳. نگاهی به جنبه‌های احتمالی

چند کلمه‌ی مرتبط با جنبه‌های محصول کنترلی را جدا نگاه می‌کنیم. اگر کلمه‌ای مثل `battery` در نقدهای منفی خیلی بیشتر باشد، نشانه‌ی اولیه‌ای است که آن جنبه با نارضایتی همراه است؛ همان چیزی که گزارش ما در فاز ۴ می‌خواهد اندازه بگیرد.

</div>

In [ ]:
aspect_words = ["battery", "charge", "charger", "broke", "broken", "range", "signal",
                "remote", "controller", "motor", "propeller", "camera", "durable",
                "speed", "fast", "instructions", "price", "shipping", "kids", "fun"]
words.reindex([a for a in aspect_words if a in words.index]).sort_values("log2_ratio", ascending=False)

<div dir="rtl">

## ۵. جمع‌بندی

_این بخش را بعد از اجرای نوت‌بوک و دیدن خروجی‌ها با هم می‌نویسیم._

</div>